# Direct Ptychography Kernels in `quantEM`

Tutorial by Georgios Varnavides (`G.Varnavides@tudelft.nl`), TU Delft and Stephanie Ribet, LBNL (`sribet@lbl.gov`)

## What this notebook does

Direct ptychography reconstructs an object by deconvolving the effect of the probe from a 4D-STEM dataset. There are several ways to build that deconvolution, and in `quantEM` swapping between them is a single keyword. This notebook uses a *simulated* strontium titanate dataset, where we know the aberrations and the dose exactly, to see what each choice actually buys you.

By the end of the notebook you will have:

1. Loaded a simulated, Nyquist-sampled 4D-STEM dataset of SrTiO$_3$ acquired under three different aberration conditions.
2. Applied a finite electron dose and seen what shot noise does to the raw patterns.
3. Reconstructed the same data with all five deconvolution kernels and compared them.
4. Seen how aberrations and dose change which kernel you should reach for.
5. Recovered resolution from a deliberately under-sampled scan using `upsampling_factor`.

Everything here runs comfortably on a CPU, so no GPU runtime is needed.

## 1. Set Up the Environment

Install `quantEM` and import what we need. On Colab the install takes a couple of minutes, so start it now and read ahead while you wait.

In [ ]:
%pip install -q git+https://github.com/electronmicroscopy/quantem.git@dev

In [ ]:
import numpy as np
import quantem as em

## 2. Download the Data

The dataset is a multislice simulation of a thin SrTiO$_3$ crystal viewed along [100], recorded at 300 keV with a 20 mrad convergence semi-angle and sampled at the Nyquist limit.

It contains **three aberration settings** stacked along the first axis:

| index | condition | aberrations |
| --- | --- | --- |
| 0 | in-focus | none |
| 1 | defocus | `C10` = 50 Å |
| 2 | defocus + coma | `C10` = 50 Å, `C21` = 5000 Å |

The remaining axes are the 32 × 32 scan grid and the 112 × 112 detector.

In [ ]:
import os
import gdown

dirpath = "/content/"
filepath_data = dirpath + "STO_20mrad_Nyquist_3aberrations.zip"

if not os.path.exists(filepath_data):
    gdown.download(
        id="1TbtA4RcRsGLXTFc43v8suo00wK4g7D_f",
        output=filepath_data,
        quiet=False,
    )

In [ ]:
dataset = em.io.load(filepath_data)
dataset

In [ ]:
energy = 300e3
semiangle_cutoff = 20

# the aberrations used in the simulation, one dictionary per index along the first axis
aberrations = [
    {},                                        # in-focus
    {"C10": 50.0, "phi21": -0.8},              # defocus
    {"C10": 50.0, "C21": 5000.0, "phi21": -0.8},  # defocus + coma
]
labels = ["in-focus", "defocus", "defocus + coma"]

## 3. Apply a Finite Electron Dose

The simulated patterns are noise-free probability distributions. Real measurements are counts, so we draw Poisson samples at a chosen fluence. The helper below converts a fluence in electrons per Å$^2$ into electrons per probe position using the scan sampling.

We will work at two doses throughout: a generous 10$^6$ e$^-$/Å$^2$ and a much more realistic 10$^4$ e$^-$/Å$^2$.

In [ ]:
def add_poisson_noise(dataset, electrons_per_area):
    """Draw Poisson counts at a given fluence. `np.inf` returns the noiseless data."""
    if electrons_per_area == np.inf:
        return dataset

    electrons_per_probe = electrons_per_area * dataset.sampling[:2].prod()

    dataset_noisy = dataset.copy()
    dataset_noisy.array = np.random.poisson(dataset.array * electrons_per_probe)
    return dataset_noisy

In [ ]:
np.random.seed(2026)

noisy_datasets = [
    [add_poisson_noise(dataset[index], dose) for dose in (1e6, 1e4)]
    for index in range(3)
]

In [ ]:
em.visualization.show_2d(
    [
        noisy_datasets[0][0][0, 0].array,
        noisy_datasets[0][0].mean((0, 1)),
        noisy_datasets[0][1][0, 0].array,
        noisy_datasets[0][1].mean((0, 1)),
    ],
    title=[
        "example 1e6 pattern", "mean 1e6 pattern",
        "example 1e4 pattern", "mean 1e4 pattern",
    ],
    power=0.5,
    cmap="turbo",
);

A single low-dose pattern looks almost like noise, yet the mean over the scan is perfectly well behaved. That gap is the whole point of ptychography: the information is spread across the scan, not concentrated in any one pattern.

## 4. Reconstruct with Each Deconvolution Kernel

Build a `DirectPtychography` object from the 4D dataset. At this stage we supply the beam energy, the convergence semi-angle, the scan-to-detector rotation (zero, because this is a simulation), and the known aberrations.

In [ ]:
direct_ptycho = em.diffractive_imaging.DirectPtychography.from_dataset4d(
    noisy_datasets[1][0],           # defocus, high dose
    energy=energy,
    semiangle_cutoff=semiangle_cutoff,
    rotation_angle=0,
    aberration_coefs=aberrations[1],
    verbose=False,
)

`quantEM` implements five deconvolution kernels, and each is one keyword away:

- **`'ssb'`** — the double-overlap weighted kernel of single-sideband ptychography. Aliases: `'single-sideband'`, `'aberration-corrected-bright-field'`, `'acbf'`.
- **`'obf'`** — the noise-normalizing optimum bright-field STEM kernel. Alias: `'optimum-bright-field'`.
- **`'mf'`** — the least-squares matched filter, closely related to Wigner distribution deconvolution. Alias: `'matched-filter'`.
- **`'prlx'`** — the parallax approximation. Aliases: `'parallax'`, `'tilt-corrected-bright-field'`, `'tcbf'`.
- **`'icom'`** — the first moment of the direct-ptychography gamma kernel. Alias: `'center-of-mass'`.

The scan here covers a single SrTiO$_3$ unit cell, so we tile each reconstruction 2 × 2 to make the lattice easier to read.

In [ ]:
kernels = ["ssb", "obf", "mf", "prlx", "icom"]

recons = [
    np.tile(direct_ptycho.reconstruct(deconvolution_kernel=k, verbose=False).obj, (2, 2))
    for k in kernels
]

em.visualization.show_2d(
    recons,
    title=["single-sideband", "optimum-bright-field", "matched-filter",
           "parallax", "center-of-mass"],
    norm="minmax",
    axsize=(3, 3),
);

All five recover the same lattice, which is reassuring, but they differ in contrast and in how they weight spatial frequencies. `'icom'` in particular is the one you already met on the [center-of-mass imaging page](https://curiousbeams.github.io/workshop-20260810-iucr-4dstem/differential-phase-contrast): it is the same measurement, arrived at through the direct-ptychography formalism.

There is a convenience wrapper, `_reconstruct_all_permutations()`, that runs all five at once. We use it from here on.

## 5. The Effect of Aberrations and Dose

Now run the same comparison across all six combinations of the three aberration settings and the two doses. Watch which kernels degrade first.

In [ ]:
kwargs = {
    "title": ["single-sideband", "optimum-bright-field", "matched-filter",
              "parallax", "center-of-mass"],
    "norm": "minmax",
    "axsize": (3, 3),
}


def compare_kernels(index, dose_index, **extra):
    """Reconstruct one (aberration, dose) combination with all five kernels."""
    ptycho = em.diffractive_imaging.DirectPtychography.from_dataset4d(
        noisy_datasets[index][dose_index],
        energy=energy,
        semiangle_cutoff=semiangle_cutoff,
        rotation_angle=0,
        aberration_coefs=aberrations[index],
        verbose=False,
    )
    recons = ptycho._reconstruct_all_permutations(verbose=False, **extra)
    em.visualization.show_2d([np.tile(r, (2, 2)) for r in recons], **kwargs)

In [ ]:
# in-focus
compare_kernels(0, 0)   # high dose
compare_kernels(0, 1)   # low dose

In [ ]:
# defocus
compare_kernels(1, 0)   # high dose
compare_kernels(1, 1)   # low dose

In [ ]:
# defocus + coma
compare_kernels(2, 0)   # high dose
compare_kernels(2, 1)   # low dose

Two things are worth noticing.

**Aberrations are not fatal, as long as you know them.** Because the aberration function enters the deconvolution kernel explicitly, a defocused or comatic probe still reconstructs cleanly. This is a genuine strength of direct ptychography over conventional imaging, and it is why the next notebook spends its time *estimating* those aberrations when they are unknown.

**Dose separates the kernels.** At 10$^6$ e$^-$/Å$^2$ the five results are nearly interchangeable. At 10$^4$ the differences are obvious, and the noise-weighted kernels hold up better than the naive ones. This is exactly the regime where beam-sensitive samples live.

## 6. Upsampling a Sub-Sampled Scan

Real scans are often coarser than Nyquist, because scanning takes time and dose. Direct ptychography can recover some of the lost information, because each diffraction pattern samples a range of spatial frequencies rather than a single point.

Below we throw away three quarters of the scan positions (`::2` in both directions), then reconstruct with `upsampling_factor=2` to put the object back on the original grid.

In [ ]:
subsampled = add_poisson_noise(dataset[1, ::2, ::2], 1e4)
subsampled

In [ ]:
subsampled_ptycho = em.diffractive_imaging.DirectPtychography.from_dataset4d(
    subsampled,
    energy=energy,
    semiangle_cutoff=semiangle_cutoff,
    rotation_angle=0,
    aberration_coefs=aberrations[1],
    verbose=False,
)

recons = subsampled_ptycho._reconstruct_all_permutations(verbose=False)
em.visualization.show_2d([np.tile(r, (2, 2)) for r in recons], **kwargs);

Without upsampling, the reconstruction lives on the coarse scan grid and the lattice is barely sampled. Now ask for twice the sampling:

In [ ]:
recons = subsampled_ptycho._reconstruct_all_permutations(
    upsampling_factor=2,
    verbose=False,
)
em.visualization.show_2d([np.tile(r, (2, 2)) for r in recons], **kwargs);

The lattice reappears. Note that `'icom'` is the exception: the center-of-mass kernel throws away the information upsampling relies on, so it cannot be upsampled.

The resolution ceiling is set by the aperture, not by the scan: direct ptychography can reach twice the convergence semi-angle, no matter how finely you upsample beyond that.

## 7. What to Notice and What to Try Next

The practical summary:

- Swapping deconvolution kernels is one keyword, so try several. They agree at high dose and disagree in exactly the regime you care about.
- Known aberrations are handled by the kernel. Unknown aberrations are the actual problem, and the subject of the next notebook.
- `upsampling_factor` recovers real information from a sub-sampled scan, up to the 2$\alpha$ ceiling, for every kernel except `'icom'`.

Follow-up exercises:

1. Push the dose lower, to 10$^3$ or 10$^2$ e$^-$/Å$^2$, and find where each kernel breaks.
2. Reconstruct the defocused dataset while *pretending* it is in-focus, by passing `aberration_coefs={}`, and see how much you lose.
3. Sub-sample by `::3` or `::4` and see how far `upsampling_factor` can be pushed.
4. Compare `'obf'` against `'ssb'` at low dose and read Ref. [1] on why the noise weighting helps.

## References

[1] G. Varnavides, W. P. M. de Kleijne, and S. M. Ribet, "The ABCs of phase retrieval: Connecting the acronyms of scanning transmission electron microscopy," *MRS Bulletin* (2026). DOI: <https://doi.org/10.1557/s43577-026-01100-3>.

[2] `quantEM` documentation: <https://electronmicroscopy.github.io/quantem-docs/>